# Model experiment

This notebook is where the production model is chosen. It calls `medilinda_ml.experiment.run_experiment`, the same function as `python -m medilinda_ml.experiment`.

The experiment does four things:

1. It splits the data. The test rows stay untouched.
2. It selects the features with recursive feature elimination. SMOTE runs inside each fold.
3. It searches the grid of each of the 8 candidates with cross-validation on the training rows.
4. It picks the candidate with the highest mean cross-validation weighted F1, and writes `selection.json`.

MLflow records each step in `server/mlruns`. The production training (`train_model.py`) reads `selection.json`. So the model in production is the model chosen here.

Start the kernel with `PYTHONPATH` set to `server/src`. Run the notebook from any folder.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from medilinda_ml.data_split import split_dataset
from medilinda_ml.experiment import run_experiment
from medilinda_ml.paths import DATA_DIR, SELECTION_PATH
from medilinda_ml.tracking import portable_runs_folder

sns.set_theme(style="whitegrid")

## 1. The data and the split

The split is the one that the training uses, with the same seed.

In [ ]:
df = pd.read_csv(DATA_DIR / "data.csv")
split = split_dataset(df=df)

pd.Series(
    {"train": len(split.X_train), "validation": len(split.X_val), "test": len(split.X_test)},
    name="rows",
).to_frame()

## 2. Run the experiment

This cell takes several minutes. It writes `selection.json` and adds runs to `server/mlruns`.
The test rows are not used here.

In [ ]:
with portable_runs_folder():
    result = run_experiment()

## 3. The selected features

These are the columns that the model uses. The experiment found them with cross-validation on the training rows.

In [ ]:
pd.Series(result.selected_features, name="feature").to_frame()

## 4. The candidates

Each row is the best setting that the search found for one candidate. The score is the mean weighted F1 over the cross-validation folds of the training rows.
The validation score is for information. It does not choose the winner.

In [ ]:
comparison = (
    pd.DataFrame([candidate.model_dump() for candidate in result.candidate_results])
    .sort_values("cv_mean_weighted_f1", ascending=False)
    .reset_index(drop=True)
)
comparison[["name", "cv_mean_weighted_f1", "cv_std_weighted_f1", "validation_weighted_f1", "best_params"]]

In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(
    comparison["name"],
    comparison["cv_mean_weighted_f1"],
    xerr=comparison["cv_std_weighted_f1"],
)
plt.gca().invert_yaxis()
plt.xlabel("Mean cross-validation weighted F1 (training rows)")
plt.title("Candidates")
plt.tight_layout()
plt.show()

## 5. The winner

The winner is written to `selection.json`, with the hash of the data and the id of the MLflow run.

In [ ]:
print(result.winner.name, result.winner.best_params)
print(SELECTION_PATH.read_text())

## 6. From here to production

1. Commit `selection.json` and the new runs in `server/mlruns`.
2. Run `PYTHONPATH=src uv run --group train python -m medilinda_ml.modeling.train_model` from the `server` folder. It reads `selection.json`, trains that model, and evaluates it once on the test rows.
3. To look at the runs: `mlflow ui --backend-store-uri mlruns`, from the `server` folder.